# PPO: Proximal Policy Optimization

## Why PPO Exists

## Core Mechanism

We already know policy gradients and actor-critic, so PPO starts from the problems they have.

### 1. The Problem: Policy Updates Can Be Too Large

Suppose the old policy gives:

$$\pi_{\text{old}}(\text{action} \mid \text{state}) = 0.50$$

We collect a trajectory and determine that this action had **positive advantage**.

A normal policy-gradient update increases its probability:

$$0.50 \to 0.60 \to 0.70 \to \ldots$$

**The Problem:** A single batch of experience can cause the policy to move **too far**.

A large policy change can make the new policy very different from the policy that generated the data.

**PPO's Solution:** Constrain this change.

### 2. PPO Compares Old and New Policies

For every action from the collected trajectory, calculate:

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$$

This is simply the **probability ratio**.

**Example:**

```
old probability = 0.50
new probability = 0.60
ratio = 0.60 / 0.50 = 1.2
```

**Interpretation:**

| Ratio | Meaning |
|---|---|
| `ratio = 1` | Probability unchanged |
| `ratio > 1` | New policy increased probability |
| `ratio < 1` | New policy decreased probability |

### 3. Combine Ratio with Advantage

We already calculate an advantage $A_t$.

- If $A > 0$ → the action was better than expected, so we want its probability to **increase**
- If $A < 0$ → the action was worse than expected, so we want its probability to **decrease**

The basic policy objective is:

$$r_t A_t$$

**Example:**

```
ratio = 1.2
advantage = +2
objective = 1.2 × 2 = 2.4
```

*So far, this is just importance-weighted policy gradient.*

### 4. PPO Clipping

PPO introduces a **clipping range**:

$$1 - \epsilon \leq r_t \leq 1 + \epsilon$$

Usually: $\epsilon = 0.2$

So the useful range is:

```
0.8 ───────── 1.0 ───────── 1.2
```

**Example:**

```
old probability = 0.50
new probability = 0.80
ratio = 1.6
```

If the advantage is positive, ordinary policy gradient would strongly reward this increase.

**PPO clips the ratio:**

$$1.6 \to 1.2$$

The policy therefore doesn't receive additional objective benefit from moving further in that direction.

### 5. Why the Min Exists

The actual PPO surrogate objective is:

$$L^{\text{CLIP}} = \mathbb{E} \left[ \min \left( r_t A_t, \operatorname{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t \right) \right]$$

The `min` makes PPO **conservative**:

| Advantage | Goal | Constraint |
|---|---|---|
| **Positive** | Increase probability | But not excessively |
| **Negative** | Decrease probability | But not excessively |

**Why:** Clipping limits how much improvement the optimizer can claim from moving the policy too far.

### 6. Why PPO Keeps the Old Policy

This is important.

We collect data using: $\pi_{\text{old}}$

Then **freeze those probabilities**.

**Example:** Suppose our rollout contained:

```
state = S
action = RIGHT
π_old(RIGHT|S) = 0.40
```

During optimization, the current policy may change:

```
π_new(RIGHT|S) = 0.44  (ratio = 1.10)
π_new(RIGHT|S) = 0.50  (ratio = 1.25)
π_new(RIGHT|S) = 0.55  (ratio = 1.375)
```

We **always compare against the original: 0.40**

**Why:** The ratio tells PPO how far the current policy has moved from the policy that generated this data.

That's why we **store the old log-probabilities during rollout**.

### 7. PPO Training Loop

The complete mechanism is:

```
                OLD POLICY
                    │
                    ▼
             collect rollout
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
       rewards            old log_probs
          │
          ▼
       GAE / advantages
          │
          ▼
    ┌─────────────────┐
    │ PPO optimization│
    └─────────────────┘
          │
          ▼
 current log_probs
          │
          ▼
 ratio = exp(new_log_prob - old_log_prob)
          │
          ▼
      clipping
          │
          ▼
      policy loss
          │
          ▼
      update actor
```

The **critic** is trained alongside it using the value targets.

### 8. One Subtle but Important Point

**PPO does NOT prevent the parameters from changing by more than 20%.**

The 0.2 clipping applies to the **probability ratio in the PPO objective**, not directly to neural-network weights.

**That distinction matters.**

## Q&A: Is the Critic Independent of Actions?

**Yes** — the critic $V(s)$ is **independent of the specific action**.

### Critic vs Actor

**The Critic evaluates the state itself:**

$$V(s) = \text{"How good is it to be in this state?"}$$

For example, $V(S_1) = 7$ doesn't care whether you choose $\uparrow$, $\rightarrow$, $\downarrow$, or $\leftarrow$.

**The Actor evaluates/chooses actions:**

$$\pi(a|S_1) = \begin{cases}
\pi(\uparrow|S_1) \\
\pi(\rightarrow|S_1) \\
\pi(\downarrow|S_1) \\
\pi(\leftarrow|S_1)
\end{cases}$$

### The Key Difference

We use the critic's **state value as the reference** to judge the **chosen action**.

| Component | Input | Output |
|---|---|---|
| **Critic** | State | Expected future reward |
| **Actor** | State | Action probabilities |

## Concrete Example: Actor-Critic in Action

We are at state $S_1$.

The critic currently predicts:

$$V(S_1) = 5$$

**Meaning:** "From $S_1$, I currently expect about 5 total future reward."

### The Actor Chooses an Action

```
S1
 ↓
actor chooses RIGHT
 ↓
environment
```

The environment gives:

- **Reward:** +2
- **Next state:** S2

### Step 1: What Does the Critic Think About S2?

The critic looks at $S_2$ and predicts:

$$V(S_2) = 6$$

**Summary so far:**

```
S1 --RIGHT / +2--> S2
V(S1) = 5
V(S2) = 6
```

### Step 2: Calculate What S1 Was Actually Worth

We don't know the true value of $S_1$ yet, so we create a **target**:

$$\text{target} = \text{immediate reward} + \gamma \times V(S_2)$$

With:
- $\text{reward} = 2$
- $\gamma = 0.9$
- $V(S_2) = 6$

$$\text{target} = 2 + 0.9 \times 6 = 7.4$$

**Interpretation:** "Based on what just happened, $S_1$ looks like it was worth about 7.4, not 5."

### Step 3: Update the Critic

**Before:**
- Critic prediction: $V(S_1) = 5$
- Target: $7.4$

**The critic's training pushes:**

$$5 \to \text{closer to } 7.4$$

That's all the critic update is doing.

### Step 4: What About the Actor?

Now we can use this information to judge the action.

The critic expected $V(S_1) = 5$

But the observed outcome suggests $7.4$

So the **Advantage** is:

$$A = 7.4 - 5 = +2.4$$

**Positive → outcome was better than the critic expected**

Therefore the actor gets feedback:

> "RIGHT turned out better than expected; increase its probability."

### The Complete Flow

```
S1
 │
 │ critic: "I expect 5"
 ↓
Actor chooses RIGHT
 │
 ↓
Environment
 │
 ├── reward = +2
 ↓
S2
 │
 │ critic: "S2 is worth 6"
 ↓
target = 2 + 0.9×6 = 7.4
 │
 ├──→ Critic: 5 → 7.4
 │
 └──→ Actor: outcome was better than expected
             → increase RIGHT probability
```

### Key Distinction

The **critic** is learning: *"How valuable is $S_1$?"*

The **actor** is learning: *"Which action should I take from $S_1$?"*

## Student Question

> **Q:** But at starting, if critic's state values themselves were random, how is it trained then purely based on next state which is again a random value? In short, how does bootstrapping work if everything starts random?

## Answer: Bootstrapping from Scratch

At initialization, the critic can be **wrong everywhere**. That's okay because **it doesn't need a correct critic to start learning**.

### The Key Anchor: Environment Reward

**The environment reward is the anchor.**

### Concrete Example

**Critic starts randomly:**

```
V(S1) = random 3
V(S2) = random 8
```

**Agent does:**

```
S1 → action → S2
```

**Environment says:**

```
reward = +2
```

**Our first target is:**

$$\text{target} = 2 + \gamma \times V(S_2) = 2 + 0.9 \times 8 = 9.2$$

Yes, 9.2 is **partly wrong** because $V(S_2) = 8$ was random.

But we still update:

$$V(S_1): 3 \to 9.2$$

### How It Works: The Propagation Chain

Crucially, **the agent keeps experiencing the environment**.

When it eventually reaches an actual **terminal reward**, there is no future estimate needed:

```
S9 → terminal
reward = +10
target = 10  ← NO BOOTSTRAPPING! This is REAL.
```

That **real reward propagates backward** through TD updates:

```
S9 → actual +10
 ↑
S8 → learns from S9
 ↑
S7 → learns from S8
 ↑
...
S1
```

### The Takeaway

- The critic **starts noisy**
- Real environmental rewards **progressively anchor** its estimates
- Once the agent encounters a real reward, TD learning bootstraps information backward
- Each corrected state becomes a better teaching signal for the previous state

## Powerful Example: Value Propagation in a Deterministic Episode

Let's use a **very simple deterministic episode** with a **single sparse reward** at the goal.

### The Episode Structure

```
S1 → S2 → S3 → S4 → GOAL
                         +10
```

**Reward structure:**

```
S1→S2 = 0
S2→S3 = 0
S3→S4 = 0
S4→GOAL = +10
```

Only the goal gives reward.

### Initial State: Everything Starts at Zero

```
V(S1) = 0
V(S2) = 0
V(S3) = 0
V(S4) = 0
```

### Episode 1, Phase 1: The Terminal Transition

```
S4 → GOAL
reward = +10
```

There is **no future state value** because we're terminal.

So: $$\text{target } V(S_4) = 10$$

**S4 learns:**

$$V(S_4): 0 \to 10$$

Already, the real reward has made a **huge change** (0 → 10).

### Phase 2: Next Time We Encounter S3

Now $S_3 \to S_4$ with reward = 0.

But $S_4$ is **no longer zero** — we just updated it!

$$V(S_4) = 10$$

Therefore:

$$\text{target}(S_3) = 0 + \gamma \times V(S_4) = 0 + \gamma \times 10$$

With $\gamma = 0.9$:

$$\text{target}(S_3) = 9$$

**S3 learns:**

$$V(S_3): 0 \to 9$$

### Phase 3: Next Time We Encounter S2

Now $S_3$ has information:

$$V(S_3) = 9$$

So:

$$\text{target}(S_2) = 0 + 0.9 \times 9 = 8.1$$

**S2 learns:**

$$V(S_2): 0 \to 8.1$$

### Phase 4: Finally S1

$$\text{target}(S_1) \approx 0.9 \times 8.1 = 7.29$$

**S1 learns:**

$$V(S_1): 0 \to 7.29$$

### What Happened: The Reward Cascaded Backward

Look at the final values:

```
              +10
               ↓
S1 → S2 → S3 → S4 → GOAL
↑     ↑     ↑     ↑
7.29  8.1   9    10
```

**Key Insight:** The +10 reward didn't need to be given at every step.

The **value estimate itself carries the information backward through TD bootstrapping**.

**That's why bootstrapping is so powerful.**

### Important Caveat: Reality Is Messier

It **isn't necessarily** one episode → perfectly propagate +10 all the way back.

With:
- Neural networks (function approximation)
- Stochastic environments
- Imperfect exploration
- TD updates with variance

Learning happens **gradually across many experiences**, not perfectly in one episode.

### The Real Problem: Sparse Rewards

Your earlier intuition was right:

> **If the reward is extremely sparse AND the agent rarely reaches it, learning can be painfully slow.**

This is a **real RL problem** called the **sparse-reward / credit-assignment problem**.

But once the agent encounters the reward, the critic can **propagate its information** through the value estimates.

### The Chain: How TD Learning Works

So the information flow is:

```
REAL REWARD
    ↓
V(S4)
    ↓
V(S3)
    ↓
V(S2)
    ↓
V(S1)
```

**Not** because the critic magically knows the answer.

**But** because **each corrected state becomes a better teaching signal for the previous state**.

**That's the piece that makes TD learning work despite sparse rewards.**

## Quick Clarification: Ratio vs Advantage

### What Is the Ratio?

$$r_t = \frac{\text{new policy probability}}{\text{old policy probability}}$$

**Correct.**

### What Is the Advantage?

**Not** simply "current value vs future reward."

The **Advantage** is roughly:

$$A_t = \underbrace{\text{actual reward} + \text{estimated future value}}_{\text{target}} - \underbrace{\text{current state's estimated value}}_{V(s)}$$

In other words:

$$A_t = \text{target} - V(s_t)$$

Where the target incorporates the **real environment reward** plus **bootstrapped estimates** of future value.